<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 05. Batch Normalization — Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift

- **원 논문:** [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://proceedings.mlr.press/v37/ioffe15.pdf)
- **저자 / 발표:** Sergey Ioffe and Christian Szegedy · ICML 2015 (2015)
- **난이도 / 예상 시간:** 중급 · 약 90분
- **선수 지식:** 평균·분산, nn.Module parameters/buffers, train/eval mode

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

Algorithm 1의 mini-batch mean/variance, normalization, 학습 가능한 γ/β를 직접 구현하고 running statistics를 갖는 module로 확장한다. 표준화 수치와 train/eval 동작을 assert하며 서로 다른 feature scale의 합성 분류에서 loss curve를 비교한다.

**원 논문과 다른 것**

원 논문은 ImageNet의 Inception network와 MNIST에서 큰 batch training, population-statistic 추론, 여러 optimization schedule을 평가한다. 여기서는 96×4 tabular mini-batches와 16-unit MLP를 사용하며 논문의 claimed speedup 자체를 benchmark하지 않는다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3, Algorithm 1 lines 1–2: μ_B와 σ_B² | `manual_batch_norm`의 mean과 biased batch variance | 정규화 결과의 feature-wise mean≈0, variance≈1 assert |
| §3, Eqs. (1)–(2)와 Algorithm 1 lines 3–4: x-hat, y=γx-hat+β | epsilon 정규화와 learnable `weight`/`bias` | 임의 γ/β를 넣은 출력 mean이 β인지 검증 |
| §3.1: inference에서 population statistics 사용 | `RunningBatchNorm1d.running_mean/running_var` buffers | eval 두 호출이 같고 running buffers가 변하지 않는지 assert |
| §3.2: BN transform 뒤 nonlinearity 적용 | `ScaledMLP`: Linear → BN → ReLU | 서로 다른 입력 scale에서 BN/no-BN loss와 정확도 plot |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\hat x_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}},
\qquad y_i=\gamma\hat x_i+\beta
$$

- $\mu_B,\sigma_B^2$는 mini-batch 통계, $\gamma,\beta$는 학습 가능한 affine parameter입니다.
- 학습 중에는 batch 통계, 평가 중에는 running 통계를 사용하는 차이가 핵심입니다.
- 직접 구한 normalization과 framework layer의 출력·gradient를 비교합니다.
- dense 입력은 feature 축, CNN 입력은 channel 축별로 통계를 계산합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3, Algorithm 1 lines 1–2: μ_B와 σ_B²
- **노트북 구현:** `manual_batch_norm`의 mean과 biased batch variance
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 정규화 결과의 feature-wise mean≈0, variance≈1 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 ImageNet의 Inception network와 MNIST에서 큰 batch training, population-statistic 추론, 여러 optimization schedule을 평가한다. 여기서는 96×4 tabular mini-batches와 16-unit MLP를 사용하며 논문의 claimed speedup 자체를 benchmark하지 않는다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. Algorithm 1을 네 줄로 분해하기

feature마다 mini-batch 평균 `μ_B`와 분산 `σ_B²`를 구하고, epsilon을 더해 정규화한
뒤 학습되는 `γ, β`로 표현력을 복원합니다. 이 실습의 batch variance는 Algorithm 1처럼
분모가 `m`인 biased variance입니다. 추론 때는 batch 자체가 아니라 학습 중 누적한
running statistics를 씁니다. 이는 단일 batch 함수와 stateful module을 따로 구현하는 이유입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** batch [B,D] 또는 [B,C,H,W] → 같은 shape의 normalized activation
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(5)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device

generator = torch.Generator().manual_seed(707)
latent_features = torch.randn(96, 4, generator=generator)
feature_scale = torch.tensor([1.0, 20.0, 0.05, 5.0])
feature_shift = torch.tensor([0.0, 3.0, -2.0, 10.0])
bn_x = latent_features * feature_scale + feature_shift
bn_y = (
    latent_features[:, 0]
    + latent_features[:, 1]
    - 0.8 * latent_features[:, 2]
    + 0.5 * latent_features[:, 3]
    > 0
).long()
bn_x, bn_y = ACCELERATOR.move(bn_x, bn_y)
assert bn_x.shape == (96, 4)
assert bn_y.unique().tolist() == [0, 1]
print(ACCELERATOR.summary())
print("raw means:", bn_x.mean(0).round(decimals=2).tolist())
print("raw stds: ", bn_x.std(0, unbiased=False).round(decimals=2).tolist())

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3, Algorithm 1 lines 1–2: μ_B와 σ_B² / §3, Eqs. (1)–(2)와 Algorithm 1 lines 3–4: x-hat, y=γx-hat+β
- **이 셀의 책임:** `manual_batch_norm`의 mean과 biased batch variance / epsilon 정규화와 learnable `weight`/`bias`
- **Tensor shape 계약:** batch [B,D] 또는 [B,C,H,W] → 같은 shape의 normalized activation
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 정규화 결과의 feature-wise mean≈0, variance≈1 assert / 임의 γ/β를 넣은 출력 mean이 β인지 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: Algorithm 1 lines 1–4를 구현해 (output, mean, variance)를 반환하세요.
def manual_batch_norm(x, gamma, beta, eps=1e-05):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3, Algorithm 1 lines 1–2: μ_B와 σ_B² / §3, Eqs. (1)–(2)와 Algorithm 1 lines 3–4: x-hat, y=γx-hat+β / §3.1: inference에서 population statistics 사용
- **이 셀의 책임:** `manual_batch_norm`의 mean과 biased batch variance / epsilon 정규화와 learnable `weight`/`bias` / `RunningBatchNorm1d.running_mean/running_var` buffers
- **Tensor shape 계약:** batch [B,D] 또는 [B,C,H,W] → 같은 shape의 normalized activation
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 정규화 결과의 feature-wise mean≈0, variance≈1 assert / 임의 γ/β를 넣은 출력 mean이 β인지 검증 / eval 두 호출이 같고 running buffers가 변하지 않는지 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: learnable weight/bias와 running_mean/running_var buffers를 갖는 BN module을 만드세요.
class RunningBatchNorm1d(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, features, eps=1e-05, momentum=0.1):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


running_bn = RunningBatchNorm1d(4).to(DEVICE)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §3.2: BN transform 뒤 nonlinearity 적용
- **이 셀의 책임:** `ScaledMLP`: Linear → BN → ReLU
- **Tensor shape 계약:** batch [B,D] 또는 [B,C,H,W] → 같은 shape의 normalized activation
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 서로 다른 입력 scale에서 BN/no-BN loss와 정확도 plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: Linear(4,16) → optional BN → ReLU → Linear(16,2) 모델을 만들고,
class ScaledMLP(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, use_batch_norm):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def train_scaled_model(use_batch_norm):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §3.2: BN transform 뒤 nonlinearity 적용
- **이 셀의 책임:** `ScaledMLP`: Linear → BN → ReLU
- **Tensor shape 계약:** batch [B,D] 또는 [B,C,H,W] → 같은 shape의 normalized activation
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 서로 다른 입력 scale에서 BN/no-BN loss와 정확도 plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: BN/no-BN loss curve를 나란히 그리고 최종 loss와 accuracy를 출력하세요.
# 이 한 seed의 toy 결과를 일반적인 speedup 증명으로 해석하지 마세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.